# JobHop v2 LLM Soft Reranking Pilot

This Colab notebook is a small preliminary experiment for an LLM reranking / score-fusion stage.

Protocol in this notebook:

1. Load JobHop from Hugging Face: `aida-ugent/JobHop`.
2. Load ESCO metadata from Hugging Face: `danieldux/ESCO`.
3. Build last-step next-occupation validation cases.
4. Use BGE-M3 to retrieve top-K candidate ESCO occupations.
5. Ask an LLM to score only the dense top candidates as plausible next occupations.
6. Do not delete candidates. Fuse dense scores with LLM plausibility scores and rerank.
7. Sweep lambda on validation only.

This is a pilot notebook only. It does not modify the cluster scripts or the main experimental protocol.


In [13]:
!pip -q install datasets pandas pyarrow requests tqdm sentence-transformers transformers accelerate openai

## Configuration

Choose one LLM backend:

- `LLM_BACKEND = "local"`: load a Hugging Face causal language model locally with `transformers`. No API key is needed.
- `LLM_BACKEND = "api"`: use an OpenAI-compatible endpoint. Set `OPENAI_API_KEY`, optionally `OPENAI_BASE_URL`, and `LLM_MODEL`.

If `RUN_LLM = False`, the notebook only runs the dense retrieval baseline.


In [14]:
import os

DATASET_ID = "aida-ugent/JobHop"
ESCO_DATASET_ID = "danieldux/ESCO"

MAX_EVAL_CASES = 50
RANDOM_SEED = 7

EMBEDDING_MODEL = "BAAI/bge-m3"
EMBED_BATCH_SIZE = 32
RETRIEVE_K = 50

# Number of dense top candidates sent to the LLM per query.
LLM_TOP_K = 20

# Soft reranking strength. Lambda is selected on validation only in this pilot.
LAMBDA_VALUES = [0.0, 0.05, 0.1, 0.2, 0.5]

RUN_LLM = True
LLM_BACKEND = os.environ.get("LLM_BACKEND", "local")  # "local" or "api"
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL")

# Used only when LLM_BACKEND = "local". This can also be a local model path.
LOCAL_LLM_MODEL = os.environ.get("LOCAL_LLM_MODEL", "Qwen/Qwen2.5-3B-Instruct")
LOCAL_LLM_MAX_NEW_TOKENS = 1024

# Optional for China/HF mirror environments, e.g. os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

METRIC_KS = [1, 5, 10]

print({
    "dataset": DATASET_ID,
    "esco_metadata": ESCO_DATASET_ID,
    "max_eval_cases": MAX_EVAL_CASES,
    "embedding_model": EMBEDDING_MODEL,
    "retrieve_k": RETRIEVE_K,
    "llm_top_k": LLM_TOP_K,
    "lambda_values": LAMBDA_VALUES,
    "run_llm": RUN_LLM,
    "llm_backend": LLM_BACKEND,
    "llm_model": LLM_MODEL,
    "local_llm_model": LOCAL_LLM_MODEL,
})


{'dataset': 'aida-ugent/JobHop', 'esco_metadata': 'danieldux/ESCO', 'max_eval_cases': 50, 'embedding_model': 'BAAI/bge-m3', 'retrieve_k': 50, 'llm_top_k': 20, 'lambda_values': [0.0, 0.05, 0.1, 0.2, 0.5], 'run_llm': True, 'llm_backend': 'local', 'llm_model': 'gpt-4o-mini', 'local_llm_model': 'Qwen/Qwen2.5-3B-Instruct'}


In [15]:
import json
import random
import re
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "cuda", torch.version.cuda, "available", torch.cuda.is_available(), "device", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))

torch 2.11.0+cu128 cuda 12.8 available True device cuda
Tesla T4


## Data Loading and JobHop v2 Preprocessing

In [16]:
SPLITS = ("train", "validation", "test")
REMOVE_UNKNOWN = True
COLLAPSE_CONSECUTIVE_DUPLICATES = True

def hf_tree(path=""):
    api = f"https://huggingface.co/api/datasets/{DATASET_ID}/tree/main"
    url = api if not path else f"{api}/{path}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()

def list_parquet_files():
    files = []
    stack = [""]
    while stack:
        path = stack.pop()
        for item in hf_tree(path):
            item_path = item.get("path") or item.get("name")
            item_type = item.get("type")
            if not item_path:
                continue
            if item_type == "directory":
                stack.append(item_path)
            elif item_path.endswith(".parquet"):
                files.append(item_path)
    return sorted(files)

def split_from_path(path):
    lower = path.lower()
    for split in SPLITS:
        if re.search(rf"(^|/|_|-)({split})(/|_|-|\.|$)", lower):
            return split
    if "val" in lower:
        return "validation"
    return None

def hf_resolve_url(path):
    return f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/{path}"

def load_split(split, split_files):
    paths = split_files.get(split, [])
    if not paths:
        raise RuntimeError(f"No parquet files found for split={split}.")
    frames = [pd.read_parquet(hf_resolve_url(path)) for path in paths]
    df = pd.concat(frames, ignore_index=True)
    print(split, df.shape, paths)
    return df

def standardize_jobhop_schema(df):
    df = df.copy()
    rename = {}
    if "resume_id" in df.columns and "person_id" not in df.columns:
        rename["resume_id"] = "person_id"
    if "university_level" in df.columns and "university_studies" not in df.columns:
        rename["university_level"] = "university_studies"
    if rename:
        df = df.rename(columns=rename)
    required = ["person_id", "matched_code"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}; available columns={list(df.columns)}")
    return df

def normalize_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() == "unknown":
        return None
    return text

def parse_year_quarter(value):
    if pd.isna(value):
        return (9999, 9)
    text = str(value)
    year_match = re.search(r"(19|20)\d{2}", text)
    year = int(year_match.group(0)) if year_match else 9999
    quarter_match = re.search(r"[Qq]([1-4])", text)
    quarter = int(quarter_match.group(1)) if quarter_match else 9
    return (year, quarter)

def make_trajectories(df):
    work = df.copy()
    work["matched_code"] = work["matched_code"].map(normalize_code)
    if REMOVE_UNKNOWN:
        work = work.dropna(subset=["matched_code"])
    work["_start_sort"] = work["start_date"].map(parse_year_quarter) if "start_date" in work.columns else [(9999, 9)] * len(work)
    work["_end_sort"] = work["end_date"].map(parse_year_quarter) if "end_date" in work.columns else [(9999, 9)] * len(work)
    sort_cols = ["person_id", "_start_sort", "_end_sort", "matched_code"]
    work = work.sort_values(sort_cols, kind="mergesort")
    trajectories = {}
    for person_id, group in work.groupby("person_id", sort=False):
        seq = []
        for _, row in group.iterrows():
            code = row["matched_code"]
            item = {
                "code": code,
                "university_studies": row.get("university_studies", None),
            }
            if COLLAPSE_CONSECUTIVE_DUPLICATES and seq and seq[-1]["code"] == code:
                continue
            seq.append(item)
        if len(seq) >= 2:
            trajectories[person_id] = seq
    return trajectories

def make_last_step_cases(trajectories):
    cases = []
    for person_id, seq in trajectories.items():
        cases.append({
            "person_id": person_id,
            "history": seq[:-1],
            "target": seq[-1]["code"],
            "university_studies": seq[-1].get("university_studies"),
        })
    return cases

In [17]:
def normalize_esco_code(value):
    if pd.isna(value):
        return None
    return str(value).strip()

def load_esco_metadata():
    frames = []
    urls = [
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/train.parquet",
        "https://huggingface.co/datasets/danieldux/ESCO/resolve/main/data/en/test.parquet",
    ]
    for url in urls:
        try:
            frames.append(pd.read_parquet(url))
        except Exception as exc:
            print("direct parquet load failed", url, exc)
    if not frames:
        ds = load_dataset("danieldux/ESCO", split="train")
        frames.append(ds.to_pandas())
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["ESCO_CODE"])
    lookup = {}
    for _, row in df.iterrows():
        code = normalize_esco_code(row.get("ESCO_CODE"))
        if code:
            lookup[code] = {
                "label": str(row.get("ESCO_OCCUPATION", "") or ""),
                "description": str(row.get("ESCO_DESCRIPTION", "") or ""),
            }
    return lookup

def occupation_label(metadata_lookup, code):
    meta = metadata_lookup.get(code, {})
    return meta.get("label") or f"ESCO code {code}"

def candidate_text(metadata_lookup, code):
    meta = metadata_lookup.get(code, {})
    label = meta.get("label") or f"ESCO code {code}"
    description = meta.get("description") or ""
    pieces = [f"ESCO occupation code: {code}.", f"Occupation title: {label}."]
    if description:
        pieces.append(f"Occupation description: {description}")
    return " ".join(pieces)

def query_text(metadata_lookup, case):
    history = "; ".join(occupation_label(metadata_lookup, item["code"]) for item in case["history"])
    codes = "; ".join(item["code"] for item in case["history"])
    edu = case.get("university_studies")
    edu_text = ""
    if edu is not None and not pd.isna(edu):
        edu_text = f" University studies: {edu}."
    return f"Career history occupations: {history}. Career history ESCO codes: {codes}.{edu_text} Predict the next ESCO occupation."

In [6]:
parquet_files = list_parquet_files()
split_files = {split: [] for split in SPLITS}
for path in parquet_files:
    split = split_from_path(path)
    if split in split_files:
        split_files[split].append(path)

raw = {split: standardize_jobhop_schema(load_split(split, split_files)) for split in SPLITS}
metadata_lookup = load_esco_metadata()
trajectories = {split: make_trajectories(raw[split]) for split in SPLITS}

candidate_codes = sorted({item["code"] for seq in trajectories["train"].values() for item in seq})
candidate_set = set(candidate_codes)
validation_cases = make_last_step_cases(trajectories["validation"])
validation_cases = [case for case in validation_cases if case["target"] in candidate_set]

rng = random.Random(RANDOM_SEED)
validation_cases = rng.sample(validation_cases, min(MAX_EVAL_CASES, len(validation_cases)))

candidate_texts = [candidate_text(metadata_lookup, code) for code in candidate_codes]
validation_query_texts = [query_text(metadata_lookup, case) for case in validation_cases]

print("candidate occupations:", len(candidate_codes))
print("validation cases:", len(validation_cases))
print("example query:", validation_query_texts[0])
print("target:", validation_cases[0]["target"], occupation_label(metadata_lookup, validation_cases[0]["target"]))

train (1594827, 5) ['JobHop_v2_train.parquet']
validation (199587, 5) ['JobHop_v2_val.parquet']
test (198877, 5) ['JobHop_v2_test.parquet']
candidate occupations: 2982
validation cases: 50
example query: Career history occupations: sales processor; administrative assistant; insurance claims handler; supply chain assistant; forwarding manager. Career history ESCO codes: 5223.5; 3343.1; 3315.3; 3343.1.7; 3331.2. University studies: Secondary school. Predict the next ESCO occupation.
target: 3343.1.7 supply chain assistant


## Dense Retrieval Baseline

In [7]:
model = SentenceTransformer(EMBEDDING_MODEL, device=device)

candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
query_embeddings = model.encode(
    validation_query_texts,
    batch_size=EMBED_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

scores = query_embeddings @ candidate_embeddings.T
top_idx = np.argsort(-scores, axis=1)[:, :RETRIEVE_K]
dense_ranked = [[candidate_codes[j] for j in row] for row in top_idx]
dense_ranked_scores = [[float(scores[i, j]) for j in row] for i, row in enumerate(top_idx)]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
def evaluate_ranked_lists(cases, ranked_lists, ks=(1, 5, 10)):
    rows = []
    n = len(cases)
    ranks = []
    for case, ranked in zip(cases, ranked_lists):
        target = case["target"]
        try:
            rank = ranked.index(target) + 1
        except ValueError:
            rank = None
        ranks.append(rank)
    result = {"n": n, "MRR": float(np.mean([1.0 / r if r else 0.0 for r in ranks]))}
    for k in ks:
        result[f"Recall@{k}"] = float(np.mean([1.0 if r and r <= k else 0.0 for r in ranks]))
    return result, ranks

baseline_metrics, baseline_ranks = evaluate_ranked_lists(validation_cases, dense_ranked, METRIC_KS)
baseline_metrics

{'n': 50,
 'MRR': 0.08801684370548468,
 'Recall@1': 0.06,
 'Recall@5': 0.12,
 'Recall@10': 0.18}

## LLM Plausibility Scoring

For each query, the LLM receives the dense top candidates and assigns each candidate a score:

- `0`: not plausible
- `1`: weakly plausible
- `2`: plausible
- `3`: highly plausible

Unlike the previous hard-filtering pilot, this notebook does not remove candidates. The scores are used as a soft reranking bonus.


In [9]:
def compact_candidate_doc(code):
    meta = metadata_lookup.get(code, {})
    label = meta.get("label") or f"ESCO code {code}"
    description = meta.get("description") or ""
    if len(description) > 260:
        description = description[:260] + "..."
    return {"code": code, "title": label, "description": description}

def build_llm_prompt(query, candidates):
    candidate_payload = [compact_candidate_doc(code) for code in candidates]
    return f"""
You are evaluating next-occupation plausibility for ESCO occupations.

Given a person's career history query and a list of candidate ESCO occupations, assign each candidate a plausibility score as the person's next occupation.

Score scale:
0 = not plausible or unrelated
1 = weakly plausible
2 = plausible
3 = highly plausible

Return only a valid JSON object whose keys are ESCO codes and whose values are integers from 0 to 3. Do not include explanations.

Career history query:
{query}

Candidate ESCO occupations:
{json.dumps(candidate_payload, ensure_ascii=False)}
""".strip()

def parse_json_object(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def score_candidates_with_openai_compatible(query, candidates, max_retries=3):
    from openai import OpenAI
    client_kwargs = {}
    if OPENAI_BASE_URL:
        client_kwargs["base_url"] = OPENAI_BASE_URL
    client = OpenAI(**client_kwargs)
    prompt = build_llm_prompt(query, candidates)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": "You are a strict JSON-only classifier for occupation plausibility."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0,
            )
            content = response.choices[0].message.content
            raw = parse_json_object(content)
            return {code: int(raw.get(code, 0)) for code in candidates}
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)

local_llm_tokenizer = None
local_llm_model = None

def load_local_llm():
    global local_llm_tokenizer, local_llm_model
    if local_llm_tokenizer is not None and local_llm_model is not None:
        return local_llm_tokenizer, local_llm_model
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    local_llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL, trust_remote_code=True)
    local_llm_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_LLM_MODEL,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    local_llm_model.eval()
    return local_llm_tokenizer, local_llm_model

def score_candidates_with_local_llm(query, candidates):
    tokenizer, llm = load_local_llm()
    prompt = build_llm_prompt(query, candidates)
    messages = [
        {"role": "system", "content": "You are a strict JSON-only classifier for occupation plausibility."},
        {"role": "user", "content": prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in messages]) + "\nASSISTANT:"
    inputs = tokenizer(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=LOCAL_LLM_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    content = tokenizer.decode(generated_ids, skip_special_tokens=True)
    raw = parse_json_object(content)
    return {code: int(raw.get(code, 0)) for code in candidates}

def score_candidates(query, candidates):
    if LLM_BACKEND == "api":
        return score_candidates_with_openai_compatible(query, candidates)
    if LLM_BACKEND == "local":
        return score_candidates_with_local_llm(query, candidates)
    raise ValueError(f"Unsupported LLM_BACKEND={LLM_BACKEND!r}; expected 'api' or 'local'.")



In [10]:
llm_scores_per_case = []
failures = []

if RUN_LLM:
    if LLM_BACKEND == "api" and not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY for api backend, switch LLM_BACKEND='local', or set RUN_LLM = False.")
    if LLM_BACKEND == "local":
        load_local_llm()
    for i, (query, ranked) in enumerate(tqdm(list(zip(validation_query_texts, dense_ranked)), desc="LLM scoring")):
        llm_candidates = ranked[:LLM_TOP_K]
        try:
            llm_scores = score_candidates(query, llm_candidates)
        except Exception as exc:
            failures.append({"case_index": i, "error": repr(exc)})
            llm_scores = {code: 0 for code in llm_candidates}
        llm_scores_per_case.append(llm_scores)
else:
    print("RUN_LLM is False; skipping LLM scoring.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM scoring:   0%|          | 0/50 [00:00<?, ?it/s]

## Soft Score Fusion and Validation Lambda Sweep

Dense top-50 candidates are never removed. For candidates in `LLM_TOP_K`, the notebook adds a normalized LLM plausibility bonus:

`final_score = dense_score_norm + lambda * (llm_score / 3)`

Candidates outside `LLM_TOP_K` keep only the dense score. Lambda is chosen using validation MRR only.


In [11]:
def minmax(values):
    arr = np.asarray(values, dtype="float32")
    lo = float(arr.min())
    hi = float(arr.max())
    if hi <= lo:
        return np.ones_like(arr, dtype="float32")
    return (arr - lo) / (hi - lo)

def rerank_with_llm_bonus(ranked_codes, dense_scores_for_row, llm_scores, lambda_value):
    dense_norm = minmax(dense_scores_for_row)
    combined = []
    for code, dense_score_norm in zip(ranked_codes, dense_norm):
        llm_score_norm = float(llm_scores.get(code, 0)) / 3.0
        final_score = float(dense_score_norm) + float(lambda_value) * llm_score_norm
        combined.append((code, final_score))
    combined.sort(key=lambda item: item[1], reverse=True)
    return [code for code, _ in combined]

def build_reranked_lists(lambda_value):
    outputs = []
    for ranked, row_scores, llm_scores in zip(dense_ranked, dense_ranked_scores, llm_scores_per_case):
        outputs.append(rerank_with_llm_bonus(ranked, row_scores, llm_scores, lambda_value))
    return outputs

def gold_in_llm_window_rate(cases, ranked_lists, top_k):
    return float(np.mean([case["target"] in ranked[:top_k] for case, ranked in zip(cases, ranked_lists)]))

def gold_llm_score_stats(cases, ranked_lists, llm_scores_per_case, top_k):
    scores = []
    for case, ranked, llm_scores in zip(cases, ranked_lists, llm_scores_per_case):
        target = case["target"]
        if target in ranked[:top_k]:
            scores.append(float(llm_scores.get(target, 0)))
    if not scores:
        return {"gold_llm_score_mean": None, "gold_llm_score_n": 0}
    return {"gold_llm_score_mean": float(np.mean(scores)), "gold_llm_score_n": len(scores)}

if RUN_LLM:
    sweep_rows = []
    gold_window = gold_in_llm_window_rate(validation_cases, dense_ranked, LLM_TOP_K)
    gold_score_stats = gold_llm_score_stats(validation_cases, dense_ranked, llm_scores_per_case, LLM_TOP_K)
    for lambda_value in LAMBDA_VALUES:
        reranked = build_reranked_lists(lambda_value)
        metrics, _ = evaluate_ranked_lists(validation_cases, reranked, METRIC_KS)
        sweep_rows.append({
            "setting": "dense_plus_llm_soft_rerank",
            "lambda": lambda_value,
            **metrics,
            "gold_in_llm_top_k": gold_window,
            "llm_top_k": LLM_TOP_K,
            "llm_failures": len(failures),
            "llm_failure_rate": len(failures) / max(1, len(validation_cases)),
            **gold_score_stats,
        })
    sweep = pd.DataFrame(sweep_rows)
    baseline_row = {"setting": "dense_baseline", "lambda": 0.0, **baseline_metrics}
    comparison = pd.concat([pd.DataFrame([baseline_row]), sweep], ignore_index=True)
    display(comparison)
    best_idx = sweep["MRR"].astype(float).idxmax()
    best_lambda = float(sweep.loc[best_idx, "lambda"])
    print("Selected lambda by validation MRR:", best_lambda)
    print("LLM failures:", len(failures))
else:
    display(pd.DataFrame([{ "setting": "dense_baseline", **baseline_metrics }]))


,setting,lambda,n,MRR,Recall@1,Recall@5,Recall@10,gold_in_llm_top_k,llm_top_k,llm_failures,llm_failure_rate,gold_llm_score_mean,gold_llm_score_n
0,dense_baseline,0.00,50,0.088017,0.06,0.12,0.18,NaN,NaN,NaN,NaN,NaN,NaN
1,dense_plus_llm_soft_rerank,0.00,50,0.088017,0.06,0.12,0.18,0.2,20.0,5.0,0.1,1.6,10.0
2,dense_plus_llm_soft_rerank,0.05,50,0.092779,0.06,0.12,0.18,0.2,20.0,5.0,0.1,1.6,10.0
3,dense_plus_llm_soft_rerank,0.10,50,0.094985,0.06,0.12,0.20,0.2,20.0,5.0,0.1,1.6,10.0
4,dense_plus_llm_soft_rerank,0.20,50,0.100128,0.06,0.12,0.20,0.2,20.0,5.0,0.1,1.6,10.0
5,dense_plus_llm_soft_rerank,0.50,50,0.103461,0.06,0.14,0.20,0.2,20.0,5.0,0.1,1.6,10.0


Selected lambda by validation MRR: 0.5
LLM failures: 5


## Inspect Examples and Save Outputs


In [12]:
if RUN_LLM:
    selected_reranked = build_reranked_lists(best_lambda)
    _, selected_ranks = evaluate_ranked_lists(validation_cases, selected_reranked, METRIC_KS)
    rows = []
    for i, case in enumerate(validation_cases):
        target = case["target"]
        rows.append({
            "case_index": i,
            "person_id": case["person_id"],
            "target_code": target,
            "target_label": occupation_label(metadata_lookup, target),
            "baseline_rank": baseline_ranks[i],
            "reranked_rank": selected_ranks[i],
            "gold_in_llm_top_k": target in dense_ranked[i][:LLM_TOP_K],
            "gold_llm_score": llm_scores_per_case[i].get(target),
            "query": validation_query_texts[i],
            "dense_top10": dense_ranked[i][:10],
            "reranked_top10": selected_reranked[i][:10],
            "llm_scores": llm_scores_per_case[i],
        })
    details = pd.DataFrame(rows)
    display(details.head(10))
    comparison.to_csv("jobhop_llm_reranking_pilot_summary.csv", index=False)
    details.to_csv("jobhop_llm_reranking_pilot_details.csv", index=False)
    with open("jobhop_llm_reranking_pilot_failures.json", "w", encoding="utf-8") as f:
        json.dump(failures, f, indent=2, ensure_ascii=False)
    print("Saved jobhop_llm_reranking_pilot_summary.csv")
    print("Saved jobhop_llm_reranking_pilot_details.csv")
    print("Saved jobhop_llm_reranking_pilot_failures.json")


,case_index,person_id,target_code,target_label,baseline_rank,reranked_rank,gold_in_llm_top_k,gold_llm_score,query,dense_top10,reranked_top10,llm_scores
0,0,148104,3343.1.7,supply chain assistant,3.0,2.0,True,3.0,Career history occupations: sales processor; a...,"[5223.5, 4311.2, 3343.1.7, 2423.1, 3343.1, 333...","[5223.5, 3343.1.7, 4311.2, 3331.2, 5223.4, 242...","{'5223.5': 3, '4311.2': 2, '3343.1.7': 3, '242..."
1,1,430074,3343.2,editorial assistant,NaN,NaN,False,NaN,Career history occupations: factory hand; inve...,"[9329.1, 4321.1, 2423.1, 1324.8.1, 4321.1.5, 2...","[9329.1, 4321.1, 4321.1.5, 1324.8, 1321.2.1.2,...","{'9329.1': 3, '4321.1': 3, '2423.1': 0, '1324...."
2,2,67623,5223.7.23,meat and meat products specialised seller,NaN,NaN,False,NaN,Career history occupations: cashier; babysitte...,"[5230.1, 5311.1.2, 5222.1.1, 2423.1, 5131.2, 2...","[5230.1, 5311.1.2, 5131.2, 5222.1.1, 5311.1, 2...","{'5230.1': 3, '5311.1.2': 3, '5222.1.1': 1, '2..."
3,3,179514,3422.4,sports instructor,NaN,NaN,False,NaN,Career history occupations: scopist; child car...,"[2635.3.2, 2423.1, 3412.4, 5311.1, 2342.1, 531...","[2635.3.2, 2423.1, 3412.4, 5311.1, 2342.1, 531...","{'2635.3.2': 3, '2423.1': 2, '3412.4': 2, '531..."
4,4,293935,4120.1,secretary,34.0,34.0,False,NaN,Career history occupations: administrative ass...,"[3343.1, 3343.1.2, 2359.1, 2423.1, 3343.3, 334...","[3343.1, 2359.1, 3343.3, 3343.4, 3343.1.2, 242...","{'3343.1': 3, '3343.1.2': 1, '2359.1': 2, '242..."
5,5,21485,5223.7.23,meat and meat products specialised seller,NaN,NaN,False,NaN,Career history occupations: cashier; accountin...,"[5230.1, 2423.1, 5222.1.1, 5223.4, 4311.2, 751...","[5230.1, 5223.4, 4311.2, 7512.3, 2423.1, 3432....","{'5230.1': 3, '2423.1': 1, '5222.1.1': 0, '522..."
6,6,32238,3321.1,insurance broker,NaN,NaN,False,NaN,Career history occupations: sports instructor;...,"[2423.1, 2424.1, 3422.4, 2120.6, 5132.1, 5132....","[2423.1, 2424.1, 2120.6, 3422.4, 5132.1.1, 522...","{'2423.1': 3, '2424.1': 2, '3422.4': 0, '2120...."
7,7,372896,3122.4.12,paper mill supervisor,NaN,NaN,False,NaN,Career history occupations: market research an...,"[2431.4, 1221.3.2, 2431.11, 1324.3.1.6.3, 1221...","[2431.4, 1221.3.2, 1324.3.1.6.3, 1324.3.1, 243...","{'2431.4': 3, '1221.3.2': 3, '2431.11': 0, '13..."
8,8,242014,8131.7,fermenter operator,NaN,NaN,False,NaN,Career history occupations: kitchen assistant;...,"[2423.1, 5120.1, 9412.1, 8331.1, 8160.34, 3139...","[2423.1, 5120.1, 9412.1, 8331.1, 8160.34, 3139...","{'2423.1': 2, '5120.1': 1, '9412.1': 0, '8331...."
9,9,41706,9321.2,hand packer,NaN,NaN,False,NaN,Career history occupations: youth offending te...,"[2635.3.27, 2635.3.26, 2635.3.27.1, 2423.1, 93...","[2635.3.27, 2635.3.26, 2635.3.27.1, 2423.1, 93...","{'2635.3.27': 3, '2635.3.26': 1, '2635.3.27.1'..."


Saved jobhop_llm_reranking_pilot_summary.csv
Saved jobhop_llm_reranking_pilot_details.csv
Saved jobhop_llm_reranking_pilot_failures.json


## How to Interpret

Useful signs:

- Best validation MRR is higher than the dense baseline.
- Recall@5 or Recall@10 improves without reducing the candidate set.
- `gold_in_llm_top_k` is reasonably high, meaning the LLM has a chance to affect the gold target.
- LLM failure rate is low.

Warning signs:

- Best lambda is `0.0`: the LLM signal is not helping.
- Metrics drop for all positive lambdas.
- many LLM failures: prompt, model size, or output parsing needs adjustment.

For a proper final experiment, choose the prompt, `LLM_TOP_K`, and lambda on validation only, then freeze them and evaluate test once.
